## Import Packages

In [0]:
# Main
import pandas as pd 
import numpy as np

# Data Processing
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Data Viz
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px


## Data Ingestion 

In [0]:
df=pd.read_csv("/Workspace/Patient Readmission Model/Expanded_Patient_Readmission_Data (1).csv")

display(df)

## Data Inspection
To understand what is contained within the raw contained

In [0]:
df.shape

In [0]:
df.columns

In [0]:
df.dtypes

In [0]:
df.isnull().sum()

In [0]:
df["Readmission"].value_counts()

In [0]:
df["Admission Type"].value_counts()

In [0]:
df["Gender"].value_counts()

In [0]:
# This line of code is to check for duplicated rows in my data
total_duplicates = df.duplicated().sum()
print(total_duplicates)

In [0]:
# This code is to check the number of unique patients in my data
df['Patient ID'].nunique()

## Data Processing

## Exploratory Data Analysis

In [0]:
df.head(1000)

In [0]:
df.columns

In [0]:
# Count distinct Patient ID by Readmission and Gender
df_count = (
    df.groupby(["Readmission", "Gender"])["Patient ID"]
      .nunique()
      .reset_index(name="Distinct Patient Count")
)

# Create grouped bar chart
fig = px.bar(
    df_count,
    x="Readmission",
    y="Distinct Patient Count",
    color="Gender",
    barmode="group",
    title="Distinct Patient Count by Readmission and Gender",
    labels={
        "Readmission": "Readmission",
        "Distinct Patient Count": "Count Distinct Patient ID",
        "Gender": "Gender"
    }
)

fig.show()

In [0]:
df.columns

In [0]:
df

## Encoding

In [0]:

# 2. Initialize the LabelEncoder
le = LabelEncoder()

# 3. Fit and transform the categorical data
df['Gender_encoded'] = le.fit_transform(df['Gender'])
df['Admission_Type_encoded']=le.fit_transform(df['Admission Type'])
df['Readmission_encoded']=le.fit_transform(df['Readmission'])

print(df)
print("\nCategory Mapping:", le.classes_)

In [0]:
display(df)

In [0]:
# # Encode Gender: Male = 1, Female = 0
# df["Gender_encoded"] = df["Gender"].map({
#     "Male": 1,
#     "Female": 0
# })

# # Encode Readmission: Yes = 1, No = 0
# df["Readmission_encoded"] = df["Readmission"].map({
#     "Yes": 1,
#     "No": 0
# })

In [0]:
df2=df[['Age', 'Length of Stay',
       'Number of Diagnoses', 'Blood Pressure','Readmission_encoded', 'Blood Sugar Levels',
       'Previous Admissions', "Gender_encoded", "Admission_Type_encoded"]]

df2.head(5)

In [0]:
# Generate matrix
matrix = df2.corr(numeric_only=True)

# Plot heatmap with values labeled inside the cells
sns.heatmap(matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.show()

In [0]:
df2.columns

### Train-Test Split

In [0]:
# 2. Specify target (y) and features (X)
# Drop the target column to isolate features
X = df2.drop(columns=['Readmission_encoded'])  

# Select only the target column
y = df2['Readmission_encoded']                 

# 3. Perform the Train-Test Split # Always split before scaling to prevent data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2,          # Allocate 20% of data for testing
    random_state=42,        # Ensures reproducibility (same split every time)
    stratify=y              # Maintains class balance (crucial for classification)
)

### Scaling

In [0]:
# Initialize and apply StandardScaler
scaler = StandardScaler()

# Fit on training data and transform both sets
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)